In [ ]:
import numpy as np
from numpy.linalg import inv, eigvals

# =========================
# 参数设置
# =========================
nA = 3                  # 群落A物种数
nB = 3                  # 群落B物种数
max_trials = 10000

# 内部作用范围（弱竞争）
within_low = -0.2
within_high = 0.0

# 跨群落作用范围（强竞争）
between_low = -1.5
between_high = -1.0

# 自限作用
self_regulation = -1.0


def generate_internal_matrix(n):
    """
    生成一个群落内部作用矩阵
    对角线固定为 self_regulation
    非对角线为较弱竞争
    """
    A = np.random.uniform(within_low, within_high, size=(n, n))
    np.fill_diagonal(A, self_regulation)
    return A


def generate_between_matrix(n_from, n_to):
    """
    生成跨群落强竞争矩阵
    """
    return np.random.uniform(between_low, between_high, size=(n_from, n_to))


def equilibrium(A, r):
    """
    求平衡点 x* = -A^{-1} r
    """
    return -inv(A) @ r


def is_locally_stable(A, x_star):
    """
    对于GLV:
        dx_i/dt = x_i(r_i + sum_j A_ij x_j)

    在平衡点处 Jacobian:
        J_ij = x_i^* A_ij
    即 J = diag(x*) @ A
    """
    J = np.diag(x_star) @ A
    eigs = eigvals(J)
    return np.all(np.real(eigs) < 0)


def invasion_growth_rate(invader_indices, resident_indices, A_full, r_full, resident_eq):
    """
    计算入侵物种在居民群落平衡下的线性增长率

    g_i = r_i + sum_j A_ij x_j^resident
    """
    g = []
    for i in invader_indices:
        growth = r_full[i]
        for idx_j, j in enumerate(resident_indices):
            growth += A_full[i, j] * resident_eq[idx_j]
        g.append(growth)
    return np.array(g)


def test_all_subset_invasions(
    invader_indices,
    resident_indices,
    A_full,
    r_full,
    resident_eq
):
    """
    对于“任意组合都无法入侵”，一个保守充分条件是：
    所有单个物种的入侵增长率都为负。
    """
    g = invasion_growth_rate(
        invader_indices,
        resident_indices,
        A_full,
        r_full,
        resident_eq
    )
    return np.all(g < 0)


success = False

for trial in range(max_trials):

    # =========================
    # 生成 r
    # =========================
    rA = np.random.uniform(0.5, 1.5, size=nA)
    rB = np.random.uniform(0.5, 1.5, size=nB)

    # =========================
    # 生成群落内部矩阵
    # =========================
    A_A = generate_internal_matrix(nA)
    A_B = generate_internal_matrix(nB)

    # =========================
    # 检查两个群落各自是否有正平衡且稳定
    # =========================
    try:
        xA = equilibrium(A_A, rA)
        xB = equilibrium(A_B, rB)
    except np.linalg.LinAlgError:
        continue

    if np.any(xA <= 0) or np.any(xB <= 0):
        continue

    if not is_locally_stable(A_A, xA):
        continue

    if not is_locally_stable(A_B, xB):
        continue

    # =========================
    # 生成跨群落强竞争矩阵
    # A_full 的结构:
    #
    # [ A_A   C_AB ]
    # [ C_BA  A_B  ]
    # =========================
    C_AB = generate_between_matrix(nA, nB)
    C_BA = generate_between_matrix(nB, nA)

    A_full = np.block([
        [A_A,   C_AB],
        [C_BA,  A_B]
    ])

    r_full = np.concatenate([rA, rB])

    # 物种索引
    A_indices = np.arange(nA)
    B_indices = np.arange(nA, nA + nB)

    # =========================
    # 检查 B 是否能入侵 A
    # =========================
    B_can_invade_A = test_all_subset_invasions(
        invader_indices=B_indices,
        resident_indices=A_indices,
        A_full=A_full,
        r_full=r_full,
        resident_eq=xA
    )

    # =========================
    # 检查 A 是否能入侵 B
    # =========================
    A_can_invade_B = test_all_subset_invasions(
        invader_indices=A_indices,
        resident_indices=B_indices,
        A_full=A_full,
        r_full=r_full,
        resident_eq=xB
    )

    # 注意：
    # test_all_subset_invasions 返回 True 表示“所有入侵增长率都为负”
    if B_can_invade_A and A_can_invade_B:
        success = True
        break

if success:
    print(f"成功找到满足条件的参数，共尝试 {trial + 1} 次")

    print("\n===== r_full =====")
    print(r_full)

    print("\n===== A_full =====")
    print(A_full)

    print("\n===== 群落A平衡 =====")
    print(xA)

    print("\n===== 群落B平衡 =====")
    print(xB)

    print("\n===== 群落A内部矩阵 A_A =====")
    print(A_A)

    print("\n===== 群落B内部矩阵 A_B =====")
    print(A_B)

    print("\n===== A -> B 跨群落矩阵 C_AB =====")
    print(C_AB)

    print("\n===== B -> A 跨群落矩阵 C_BA =====")
    print(C_BA)

else:
    print("未能在最大尝试次数内找到满足条件的参数，请提高 max_trials")